In [2]:
!wget https://github.com/GwenONERA/CyberBullying/raw/refs/heads/main/Data/CyberBullyingAdo.parquet

--2026-03-04 16:08:10--  https://github.com/GwenONERA/CyberBullying/raw/refs/heads/main/Data/CyberBullyingAdo.parquet
Resolving github.com (github.com)... 140.82.114.4
Connecting to github.com (github.com)|140.82.114.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://raw.githubusercontent.com/GwenONERA/CyberBullying/refs/heads/main/Data/CyberBullyingAdo.parquet [following]
--2026-03-04 16:08:11--  https://raw.githubusercontent.com/GwenONERA/CyberBullying/refs/heads/main/Data/CyberBullyingAdo.parquet
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 262474 (256K) [application/octet-stream]
Saving to: ‘CyberBullyingAdo.parquet’

CyberBullyingAdo.pa 100%[===================>] 256.32K  --.-KB/s    in 0.04s   

2026-03-04 16:08

In [3]:
#!/usr/bin/env python3
"""
SMS / argot toxique  →  français standard — Extracteur de paires via Gemini (Colab).

Pipeline :
  1. Charge les textes de CyberBullyingAdo.parquet à partir de la ligne START_ROW
  2. Filtre les entrées sans lettre latine ET les textes triviaux
  3. Envoie chaque texte à Gemini avec un prompt-template strict
  4. Parse les réponses JSON, stocke les paires {source, target} + contexte
  5. Détecte la polysémie, produit un CSV et un JSON déduplicé
  6. Append aux fichiers existants si reprise d'exécution
"""

import json
import os
import re
import time
import pandas as pd
from google.colab import ai

# ══════════════════════════════════════════════════════════════════════
# 0.  CONFIGURATION — MODIFIER ICI POUR REPRENDRE
# ══════════════════════════════════════════════════════════════════════
PARQUET_PATH = "./CyberBullyingAdo.parquet"

START_ROW    = 1098          # ← première ligne à traiter (0-indexed)
N_TEXTS      = 500          # nombre de lignes à traiter à partir de START_ROW
END_ROW      = START_ROW + N_TEXTS  # → lignes [100, 200)

OUTPUT_CSV   = "sms_to_french_pairs.csv"
OUTPUT_JSON  = "sms_to_french_pairs.json"
API_DELAY    = 0.4          # secondes entre chaque appel (anti rate-limit)

# Mots triviaux à exclure (le texte entier, après strip+lower, ==)
TRIVIAL_TEXTS = {"oui", "non", "salut", "bonjour"}

# ══════════════════════════════════════════════════════════════════════
# 1.  CHARGEMENT & FILTRAGE
# ══════════════════════════════════════════════════════════════════════
df = pd.read_parquet(PARQUET_PATH)
print(f"[info] Dataset complet : {len(df)} lignes")

# Tranche [START_ROW, END_ROW)
if END_ROW > len(df):
    END_ROW = len(df)
    print(f"[info] END_ROW ajusté à {END_ROW} (fin du dataset)")

df_subset = df.iloc[START_ROW:END_ROW].copy()
print(f"[info] Tranche sélectionnée : lignes {START_ROW} → {END_ROW-1}  "
      f"({len(df_subset)} lignes)")

# ── Précaution 1a : garder uniquement les textes contenant ≥ 1 lettre latine
_LATIN_RE = re.compile(
    r'[a-zA-ZàâäéèêëïîôöùûüÿçœæÀÂÄÉÈÊËÏÎÔÖÙÛÜŸÇŒÆ]'
)

def has_latin(text: str) -> bool:
    return isinstance(text, str) and bool(_LATIN_RE.search(text))

mask_latin = df_subset['TEXT'].apply(has_latin)
n_no_latin = (~mask_latin).sum()

# ── Précaution 1b : exclure les textes triviaux (mot unique sans intérêt)
def is_trivial(text: str) -> bool:
    if not isinstance(text, str):
        return True
    cleaned = text.strip().lower()
    # Supprime ponctuation terminale pour attraper "Salut." / "Oui !"
    cleaned = re.sub(r'[.!?,;:\s]+$', '', cleaned)
    cleaned = re.sub(r'^[.!?,;:\s]+', '', cleaned)
    return cleaned in TRIVIAL_TEXTS

mask_trivial = df_subset['TEXT'].apply(is_trivial)
n_trivial = mask_trivial.sum()

# Combinaison des deux filtres
df_filtered = df_subset[mask_latin & ~mask_trivial].copy()
n_removed = len(df_subset) - len(df_filtered)

print(f"[info] Filtrés → sans lettre latine : {n_no_latin}, "
      f"triviaux ({', '.join(sorted(TRIVIAL_TEXTS))}) : {n_trivial}")
print(f"[info] Après filtrage : {len(df_filtered)} textes retenus "
      f"({n_removed} écarté{'s' if n_removed > 1 else ''})\n")

# ══════════════════════════════════════════════════════════════════════
# 2.  PROMPT TEMPLATE
# ══════════════════════════════════════════════════════════════════════
KNOWN_SUSPECTS = (
    "tkt, ptn, ntm, ftg, vrm, vrmt, tjrs, srx, dsl, jte, mtn, ouf, "
    "tqt, jsp, msg, bjr, wlh, fdp, tdc, jtm, nn, bh, xd, lvdm, cc, "
    "tktt, tktp, tsais, jss, mwa, lea, çca, psq, léa, 5eu, tai, sonb, "
    "cbvn, tfk, ftp, ltn, fgtg, irl, rsa, vpn, vmt, vbn, ojn, tjr, "
    "mvs, csc, uqi, oeeee, dégouteeee, bisoussssssssssssssssssssssssssss, "
    "mdmrmrr, bk, dz, pg, sl, ls, da, ds, cq"
)

PROMPT_TEMPLATE = """\
Tu es un expert en linguistique française, spécialisé dans l'argot des \
jeunes de 12-20 ans et dans le langage toxique du cyber-harcèlement en \
ligne (réseaux sociaux, SMS, messageries).

## TÂCHE
Analyse le texte ci-dessous. Identifie **chaque** mot ou groupe de mots \
correspondant à l'un de ces cas :
1. Mal orthographié / écriture SMS / abréviation \
   (ex : "rep" → "réponds", "vrm" → "vraiment", "bcp" → "beaucoup")
2. Insulte abrégée ou codée \
   (ex : "fdp" → "fils de pute", "ntm" → "nique ta mère", "tg" → "ta gueule")
3. Verlan ou argot absent d'un dictionnaire français standard \
   (ex : "teubé" → "bête", "zer" → "laser/bizarre selon contexte")
4. Chiffre(s) substitué(s) à des lettres \
   (ex : "2m1" → "demain", "3a0" → "zéro")
5. Mot tronqué \
   (ex : "tjrs" → "toujours", "mtn" → "maintenant", "srx" → "sérieux")
6. Émoticône textuelle : traduis par sa signification émotionnelle \
   (ex : "xd" → "rire", ":)" → "sourire", "^^" → "content")
7. Onomatopée ou mot allongé : donne la forme standard \
   (ex : "bisoussss" → "bisous", "trooooop" → "trop")

## CHAÎNES SUSPECTES CONNUES (à rechercher en priorité)
{suspects}

## RÈGLES STRICTES
- Retourne **UNIQUEMENT** un tableau JSON valide. Aucun texte, aucun \
  commentaire, aucun markdown autour.
- Chaque élément : {{"source": "<mot exact du texte>", "target": "<forme standard>"}}.
- **NE traduis PAS** les mots existant dans un dictionnaire français \
  standard, même familiers : "ouais", "ok", "mec", "meuf", "truc", \
  "genre", "gars", "bof", "sympa", "mdr", "lol", "nul", "chiant", \
  "pote" sont valides — ne les corrige pas.
- **NE corrige PAS** la grammaire ni la syntaxe (accords, conjugaisons, \
  ponctuation). Corrige uniquement le **vocabulaire** SMS/abrégé/toxique.
- **Respecte le contexte** pour les mots polysémiques. \
  "sa" peut être "ça" ou le possessif "sa" : choisis selon la phrase.
- Si le texte ne contient AUCUN mot à corriger, retourne : []

## EXEMPLES IN-CONTEXT

Input: "t ou ptn tu rep jamais fdp"
Output: [{{"source": "t", "target": "t'es"}}, {{"source": "ptn", "target": "putain"}}, {{"source": "rep", "target": "réponds"}}, {{"source": "fdp", "target": "fils de pute"}}]

Input: "wlh tkt jvais le niquer srx"
Output: [{{"source": "wlh", "target": "wallah"}}, {{"source": "tkt", "target": "t'inquiète"}}, {{"source": "jvais", "target": "je vais"}}, {{"source": "srx", "target": "sérieux"}}]

Input: "non, pas trop"
Output: []

## TEXTE À ANALYSER
\"\"\"{text}\"\"\"

## SORTIE JSON :"""

# ══════════════════════════════════════════════════════════════════════
# 3.  CHARGEMENT DES RÉSULTATS EXISTANTS (REPRISE)
# ══════════════════════════════════════════════════════════════════════
existing_pairs: list[dict] = []
if os.path.exists(OUTPUT_CSV):
    try:
        existing_df = pd.read_csv(OUTPUT_CSV)
        existing_pairs = existing_df.to_dict('records')
        print(f"[info] Fichier existant '{OUTPUT_CSV}' chargé : "
              f"{len(existing_pairs)} paires précédentes\n")
    except Exception as exc:
        print(f"[warn] Impossible de charger '{OUTPUT_CSV}' : {exc}\n")

# ══════════════════════════════════════════════════════════════════════
# 4.  BOUCLE PRINCIPALE : INFÉRENCE + PARSING
# ══════════════════════════════════════════════════════════════════════
new_pairs: list[dict]    = []
failed_indices: list[int] = []
empty_count = 0

print("═" * 72)
print(f"  Traitement de {len(df_filtered)} textes via Gemini (Colab AI)")
print(f"  Plage : lignes {START_ROW} → {END_ROW - 1}")
print("═" * 72 + "\n")

for i, (row_idx, row) in enumerate(df_filtered.iterrows()):
    text = str(row['TEXT']).strip()
    if not text:
        continue

    prompt = PROMPT_TEMPLATE.format(
        text=text.replace('"', '\\"'),
        suspects=KNOWN_SUSPECTS,
    )

    try:
        response = ai.generate_text(prompt)
        raw = response.strip()

        json_match = re.search(r'\[.*?\]', raw, re.DOTALL)

        if json_match:
            pairs = json.loads(json_match.group(0))

            if not isinstance(pairs, list):
                raise ValueError("La réponse JSON n'est pas une liste")

            n_valid = 0
            for pair in pairs:
                if (isinstance(pair, dict)
                        and 'source' in pair
                        and 'target' in pair
                        and pair['source'].strip()
                        and pair['target'].strip()):
                    new_pairs.append({
                        'source':     pair['source'].strip().lower(),
                        'target':     pair['target'].strip().lower(),
                        'context':    text,
                        'text_index': int(row_idx),
                    })
                    n_valid += 1

            if n_valid:
                print(f"  [{i+1:3d}/{len(df_filtered)}] ✓ {n_valid:2d} paire(s)  "
                      f"│ {text[:60]}{'…' if len(text)>60 else ''}")
            else:
                empty_count += 1
                print(f"  [{i+1:3d}/{len(df_filtered)}] · aucun argot  "
                      f"│ {text[:60]}{'…' if len(text)>60 else ''}")

        elif '[]' in raw:
            empty_count += 1
            print(f"  [{i+1:3d}/{len(df_filtered)}] · aucun argot  "
                  f"│ {text[:60]}{'…' if len(text)>60 else ''}")
        else:
            print(f"  [{i+1:3d}/{len(df_filtered)}] ⚠ pas de JSON  "
                  f"│ réponse brute : {raw[:80]}")
            failed_indices.append(row_idx)

    except json.JSONDecodeError as exc:
        print(f"  [{i+1:3d}/{len(df_filtered)}] ✗ JSON invalide : {exc}")
        failed_indices.append(row_idx)
    except Exception as exc:
        print(f"  [{i+1:3d}/{len(df_filtered)}] ✗ Erreur API : {exc}")
        failed_indices.append(row_idx)

    time.sleep(API_DELAY)

# ══════════════════════════════════════════════════════════════════════
# 5.  FUSION AVEC LES RÉSULTATS PRÉCÉDENTS
# ══════════════════════════════════════════════════════════════════════
all_pairs = existing_pairs + new_pairs

print(f"\n{'═' * 72}")
print("  RÉSULTATS")
print("═" * 72)

pairs_df = pd.DataFrame(all_pairs)
n_pairs   = len(pairs_df)
n_new     = len(new_pairs)
n_prev    = len(existing_pairs)
n_unique  = pairs_df['source'].nunique() if n_pairs else 0

print(f"\n  Nouvelles paires (cette exécution)  : {n_new}")
print(f"  Paires précédentes (reprises)       : {n_prev}")
print(f"  Total cumulé                        : {n_pairs}")
print(f"  Termes source uniques (cumulé)      : {n_unique}")
print(f"  Textes sans argot (cette exécution) : {empty_count}")
print(f"  Échecs (parse / API, cette exec.)   : {len(failed_indices)}")

if n_pairs == 0:
    print("\n  ⚠ Aucune paire n'a été extraite — vérifier les appels API.")
    raise SystemExit

# ── Précaution 2 : Détection de polysémie ────────────────────────────
polysemy = (
    pairs_df
    .groupby('source')['target']
    .apply(lambda s: sorted(set(s)))
    .reset_index()
)
polysemy['n_targets'] = polysemy['target'].apply(len)
multi = polysemy[polysemy['n_targets'] > 1].sort_values(
    'n_targets', ascending=False
)

if len(multi):
    print(f"\n  ⚠ Termes polysémiques ({len(multi)} source(s) → cibles multiples) :")
    for _, r in multi.iterrows():
        print(f"    '{r['source']}' → {r['target']}")

# ── Table de fréquences ──────────────────────────────────────────────
freq = (
    pairs_df
    .groupby(['source', 'target'])
    .agg(count=('source', 'size'))
    .reset_index()
    .sort_values('count', ascending=False)
    .reset_index(drop=True)
)

print(f"\n  Top-30 des mappings les plus fréquents (cumulé) :\n")
with pd.option_context('display.max_colwidth', 40, 'display.width', 120):
    print(freq.head(30).to_string(index=False))

# ══════════════════════════════════════════════════════════════════════
# 6.  SAUVEGARDE (ÉCRASEMENT AVEC DONNÉES CUMULÉES)
# ══════════════════════════════════════════════════════════════════════
pairs_df.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"\n[saved] {OUTPUT_CSV}  ({n_pairs} lignes cumulées)")

# ── JSON : dictionnaire dédupliqué avec exemples de contexte ──────────
dict_entries = []
for src in pairs_df['source'].unique():
    sub = pairs_df[pairs_df['source'] == src]
    targets_agg = (
        sub.groupby('target')
        .agg(
            frequency=('target', 'size'),
            example_contexts=('context', lambda x: list(x.unique())[:3]),
        )
        .reset_index()
    )
    for _, t_row in targets_agg.iterrows():
        dict_entries.append({
            'source':           src,
            'target':           t_row['target'],
            'frequency':        int(t_row['frequency']),
            'example_contexts': t_row['example_contexts'],
        })

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(dict_entries, f, ensure_ascii=False, indent=2)
print(f"[saved] {OUTPUT_JSON}  ({len(dict_entries)} entrées)")

# ══════════════════════════════════════════════════════════════════════
# 7.  RÉSUMÉ & INDICATION POUR LA PROCHAINE EXÉCUTION
# ══════════════════════════════════════════════════════════════════════
next_start = END_ROW
print(f"\n{'═' * 72}")
print("  RÉSUMÉ")
print("═" * 72)
print(f"  • Plage traitée      : lignes {START_ROW} → {END_ROW - 1}")
print(f"  • {len(df_filtered)} textes traités (après filtrage)")
print(f"  • {n_new} nouvelles paires extraites")
print(f"  • {n_pairs} paires cumulées au total")
print(f"  • {n_unique} termes source uniques")
print(f"  • {len(multi)} termes polysémiques détectés")
print(f"  • Fichiers : {OUTPUT_CSV}, {OUTPUT_JSON}")
print(f"\n  ➜ Pour la prochaine exécution, modifier :")
print(f"      START_ROW = {next_start}")
print(f"\n✓  Pipeline terminé.\n")

[info] Dataset complet : 5608 lignes
[info] Tranche sélectionnée : lignes 1098 → 1597  (500 lignes)
[info] Filtrés → sans lettre latine : 7, triviaux (bonjour, non, oui, salut) : 3
[info] Après filtrage : 490 textes retenus (10 écartés)

════════════════════════════════════════════════════════════════════════
  Traitement de 490 textes via Gemini (Colab AI)
  Plage : lignes 1098 → 1597
════════════════════════════════════════════════════════════════════════

  [  1/490] · aucun argot  │ on trouve un terrain d'entente ?
  [  2/490] ✓  2 paire(s)  │ chui juste pour le z
  [  3/490] · aucun argot  │ Enfin elle assume
  [  4/490] ✓  1 paire(s)  │ anton65 toi t juste con
  [  5/490] · aucun argot  │ d'accord avec Sarah
  [  6/490] · aucun argot  │ C’est grâce à Sarah et theo que je m’en rends compte
  [  7/490] · aucun argot  │ J'avoue je t'aime bien en fait Fatima mais je déteste toujou…
  [  8/490] · aucun argot  │ tu dois des excuses pauline
  [  9/490] · aucun argot  │ Aucun terrain d’e